# Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

/home/seongyoonjeon/venvs/lg-aimers-hackathon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)

torch version: 2.9.1+cu130
cuda available: True
torch cuda version: 13.0


# Setting

In [ ]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 2048
MAX_SEQUENCE_LENGTH = 2048

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = [
    "embed_tokens", 
    "lm_head",
]

# ignore할 레이어 범위 설정 (# 후반 레이어 MLP → error 폭발 구간 (layer 23~29))
IGNORE_MLP_LAYERS = range(23, 30)
# IGNORE_ATTENTION_LAYERS = range(27, 30)
IGNORE_ATTENTION_LAYERS = []

for i in IGNORE_MLP_LAYERS:
    IGNORE.append(f"model.layers.{i}.mlp.gate_proj")
    IGNORE.append(f"model.layers.{i}.mlp.up_proj")

for i in IGNORE_ATTENTION_LAYERS:
    IGNORE.append(f"model.layers.{i}.self_attn.q_proj")
    IGNORE.append(f"model.layers.{i}.self_attn.k_proj")
    IGNORE.append(f"model.layers.{i}.self_attn.v_proj")
    IGNORE.append(f"model.layers.{i}.self_attn.o_proj")

# Model Loads

In [4]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    
    low_cpu_mem_usage=True, 
    # max_memory={0: "10GiB", "cpu": "20GiB"},  # GPU 메모리 여유 확보
)

print("[INFO] 모델/토크나이저 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


In [5]:
print(model)

Exaone4ForCausalLM(
  (model): Exaone4Model(
    (embed_tokens): Embedding(102400, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-29): 30 x Exaone4DecoderLayer(
        (self_attn): Exaone4Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Exaone4RMSNorm((64,), eps=1e-05)
          (k_norm): Exaone4RMSNorm((64,), eps=1e-05)
        )
        (mlp): Exaone4MLP(
          (gate_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (up_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (down_proj): Linear(in_features=4096, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Exaone4RMSNorm((2048,), eps=

# Dataset Loads & Preprocess

In [6]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [7]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

# 양자화 전 메모리 정리
import gc
torch.cuda.empty_cache()
gc.collect()

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        dampening_frac=0.005,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    
    quantization_aware_calibration=True,
    pad_to_max_length=False,
    preprocessing_num_workers=1,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=2048, max_len=2048)...


Tokenizing (num_proc=1): 100%|██████████| 2048/2048 [00:02<00:00, 690.85 examples/s]


2026-02-23T16:30:08.088009+0900 | reset | INFO - Compression lifecycle reset
2026-02-23T16:30:08.089465+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-23T16:30:08.120586+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-23T16:30:08.121198+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 2048/2048 [00:15<00:00, 133.35it/s]

2026-02-23T16:30:25.452220+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 2048 samples


...

(30/31): Calibrating: 100%|██████████| 2048/2048 [00:13<00:00, 157.43it/s]

2026-02-23T16:42:16.077473+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 2048 samples


2026-02-23T16:42:16.446456+0900 | compress | METRIC - time 0.37s
2026-02-23T16:42:16.447193+0900 | compress | METRIC - error 5800.30
2026-02-23T16:42:16.447533+0900 | compress | METRIC - GPU 0 | usage: 19.72% | total memory: 12 GB
2026-02-23T16:42:16.447854+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-23T16:42:16.448220+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 2048 samples
2026-02-23T16:42:16.809756+0900 | compress | METRIC - time 0.36s
2026-02-23T16:42:16.810526+0900 | compress | METRIC - error 1649.74
2026-02-23T16:42:16.810893+0900 | compress | METRIC - GPU 0 | usage: 19.06% | total memory: 12 GB
2026-02-23T16:42:16.811072+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-23T16:42:16.811391+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 2048 samples
2026-02-23T16:42:17.165112+0900 | compress | METRIC - time 0.35s
2026-02-23T16:42:17.165817+0900 | compress | ME

(31/31): Propagating: 100%|██████████| 2048/2048 [00:01<00:00, 1404.04it/s]

2026-02-23T16:42:27.610208+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-23T16:42:27.632743+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료


# Model Save

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-23T16:42:27.657370+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 196it [00:02, 79.00it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [ ]:
zip_name = "submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] submit-new-6.zip 생성 중...
[INFO] 생성 완료: submit-new-6.zip


# Test

In [10]:
# ── 1. 모듈 import ────────────────────────────────────────────────────────
import importlib
import evaluation

# 코드 수정 후 반영할 때 → 이 셀만 다시 실행
importlib.reload(evaluation)
print('모듈 로드 완료')


# ── 2. 평가 실행 (기본 설정) ──────────────────────────────────────────────
results = evaluation.main()


# ── 3. 평가 실행 (파라미터 직접 지정) ────────────────────────────────────
# 빠른 확인: 샘플 줄이기
results = evaluation.main(
    base_model_id  = './base_model',
    eval_model_id  = './model',
    perf_samples   = 50,    # 줄일수록 빠름
    speed_runs  = 30,
    max_len        = 512,
    max_new_tokens = 64,
)


# ── 4. 결과 확인 ──────────────────────────────────────────────────────────
print(f'PerfNorm  : {results["perf_norm"]:.4f}')
print(f'SpeedNorm : {results["speed_norm"]:.4f}')
print(f'Score     : {results["score"]:.4f}')

모듈 로드 완료
  리더보드 Score 평가 시작
  기준 모델   : ./base_model
  평가 모델   : ./model
  성능 샘플   : 100
  속도 반복   : 50회 (더미 고정 입력 len=128)
  max_len     : 512 / max_new_tokens: 64

[LOAD] Base Model (./base_model)
  GPU 사용: 2.57 GB

[DATA] 데이터 로드 중 (100개)...
  성능 평가용: 100개

[PERF] Base Perplexity 계산 중...


  PPL (Base): 100%|██████████| 100/100 [00:03<00:00, 26.19it/s]


  Perplexity: 5.6027  (skipped: 0)

[SPEED] Base 속도 측정 중 (50회 반복)...
  워밍업 중...


  Speed (Base): 100%|██████████| 50/50 [00:00<00:00, 70.71it/s]


  토큰당 시간: 14.0768 ms/token  (71.0 tok/s)

[LOAD] Eval Model (./model)


...

  GPU 사용: 1.15 GB

[PERF] Eval Perplexity 계산 중...


  PPL (Eval): 100%|██████████| 50/50 [00:04<00:00, 11.83it/s]


  Perplexity: 5.6857  (skipped: 0)

[SPEED] Eval 속도 측정 중 (30회 반복)...
  워밍업 중...


  Speed (Eval): 100%|██████████| 30/30 [00:01<00:00, 16.77it/s]


  토큰당 시간: 59.3912 ms/token  (16.8 tok/s)

  📊 평가 결과
  [Perplexity]
    Base  PPL : 5.5279
    Model PPL : 5.6857
  [Speed (ms/token)]
    Base  TPT : 14.4248 ms
    Model TPT : 59.3912 ms
    속도 배율 : 0.24x  ❌ 느림
  [Score]
    PerfNorm  : 0.9722  (기준: 1.0)
    SpeedNorm : -3.1173  (기준: 0.0)
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    🏆 Score  : 0.0000
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  총 소요 시간: 0.2분 (12.8초)

  [해석]
  ⚠️  PerfNorm 0.9722: base 대비 성능 2.8% 저하
  ⚠️  SpeedNorm -3.1173: base 대비 311.7% 느림
  🔧 Score 0.0000: 개선 여지 있음
PerfNorm  : 0.9722
SpeedNorm : -3.1173
Score     : 0.0000
